In [3]:
import yfinance as yf
import feedparser
import json
from datetime import datetime

In [4]:
def get_prices():
    tickers = ["SPY", "QQQ", "DIA", "NVDA", "SNOW"]
    data = {}
    for t in tickers:
        hist = yf.Ticker(t).history(period="2d")
        hist_dict = hist.to_dict()
        # Convert Timestamp keys to strings
        for col in hist_dict:
            hist_dict[col] = {str(k): v for k, v in hist_dict[col].items()}
        data[t] = hist_dict
    return data

def get_news():
    feeds = [
        "https://feeds.reuters.com/reuters/businessNews",
        "https://www.cnbc.com/id/100003114/device/rss/rss.html"
    ]
    articles = []
    for url in feeds:
        feed = feedparser.parse(url)
        for entry in feed.entries[:20]:
            articles.append({
                "title": entry.title,
                "summary": entry.summary,
                "link": entry.link
            })
    return articles

data = {
    "timestamp": str(datetime.now()),
    "prices": get_prices(),
    "news": get_news()
}

In [5]:
data.keys()

dict_keys(['timestamp', 'prices', 'news'])

In [6]:
with open("data/raw.json", "w") as f:
    json.dump(data, f)

In [7]:
# features.py
import json
import pandas as pd

def compute_returns(prices):
    results = {}
    for t, hist in prices.items():
        df = pd.DataFrame(hist)
        if "Close" in df:
            returns = df["Close"].pct_change().iloc[-1]
            results[t] = float(returns)
    return results

data = json.load(open("data/raw.json"))
features = {
    "returns": compute_returns(data["prices"])
}

json.dump(features, open("data/features.json", "w"))

In [8]:
# embed.py
from sentence_transformers import SentenceTransformer
import json
import faiss
import numpy as np

model = SentenceTransformer("BAAI/bge-m3")

data = json.load(open("data/raw.json"))
texts = [a["title"] + " " + a["summary"] for a in data["news"]]

embeddings = model.encode(texts, batch_size=32, show_progress_bar=True)

index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(np.array(embeddings))

faiss.write_index(index, "data/news.index")

json.dump(texts, open("data/news_texts.json", "w"))

2026-03-18 00:48:08.506211: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Skipping import of cpp extensions due to incompatible torch version 2.10.0+cu128 for torchao version 0.14.1             Please see https://github.com/pytorch/ao/issues/2919 for more info


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [9]:
# retrieve.py
import faiss, json
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("BAAI/bge-m3")

index = faiss.read_index("data/news.index")
texts = json.load(open("data/news_texts.json"))

query = "major market moving news today"
q_emb = model.encode([query])

D, I = index.search(q_emb, k=10)

top_news = [texts[i] for i in I[0]]

json.dump(top_news, open("data/context.json", "w"))

In [10]:
# summarize.py
import json
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

#"Qwen/Qwen2.5-7B-Instruct"
#"meta-llama/Meta-Llama-3-8B-Instruct"

model_name = "mistralai/Mistral-7B-Instruct-v0.3" #"meta-llama/Meta-Llama-3-8B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
    load_in_4bit = True
)

`torch_dtype` is deprecated! Use `dtype` instead!
The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [11]:
#model_name = "Qwen/Qwen2.5-7B-Instruct"

In [13]:
# signals.py
import json
import numpy as np

def generate_signals(features, news):
    signals = []

    returns = features["returns"]

    # --- Rule 1: Momentum ---
    for ticker, r in returns.items():
        if r > 0.02:
            signals.append({
                "type": "momentum_bullish",
                "ticker": ticker,
                "value": r,
                "strength": "high"
            })
        elif r < -0.02:
            signals.append({
                "type": "momentum_bearish",
                "ticker": ticker,
                "value": r,
                "strength": "high"
            })

    # --- Rule 2: Market Breadth ---
    pos = sum(1 for r in returns.values() if r > 0)
    neg = sum(1 for r in returns.values() if r < 0)

    if pos > neg * 2:
        signals.append({"type": "broad_bullish", "strength": "medium"})
    elif neg > pos * 2:
        signals.append({"type": "broad_bearish", "strength": "medium"})

    # --- Rule 3: News Sentiment (simple heuristic) ---
    negative_keywords = ["inflation", "war", "rate hike", "selloff", "recession"]
    positive_keywords = ["growth", "beat", "rally", "upgrade"]

    neg_score, pos_score = 0, 0
    for article in news:
        text = article.lower()
        neg_score += sum(k in text for k in negative_keywords)
        pos_score += sum(k in text for k in positive_keywords)

    if neg_score > pos_score * 1.5:
        signals.append({"type": "negative_news_pressure", "strength": "medium"})
    elif pos_score > neg_score * 1.5:
        signals.append({"type": "positive_news_momentum", "strength": "medium"})

    return signals


#if __name__ == "__main__":
features = json.load(open("data/features.json"))
news = json.load(open("data/context.json"))

signals = generate_signals(features, news)

json.dump(signals, open("data/signals.json", "w"), indent=2)

In [14]:
import json

def generate_alerts(signals):
    alerts = []

    for s in signals:
        if s["strength"] == "high":
            alerts.append(f"🚨 HIGH: {s}")

        elif s["type"] in ["broad_bearish", "negative_news_pressure"]:
            alerts.append(f"⚠️ RISK: {s}")

    return alerts


#if __name__ == "__main__":
signals = json.load(open("data/signals.json"))

alerts = generate_alerts(signals)

json.dump(alerts, open("data/alerts.json", "w"), indent=2)

In [15]:
# from vllm import LLM, SamplingParams

# llm = LLM(model="Qwen/Qwen2.5-7B-Instruct")

# outputs = llm.generate(prompt, SamplingParams(max_tokens=1000))

In [16]:
#pip install sendgrid -q

In [28]:
features = json.load(open("data/features.json"))
news = json.load(open("data/context.json"))
signals = json.load(open("data/signals.json"))
alerts = json.load(open("data/alerts.json"))

prompt = f"""
You are a senior macro hedge fund analyst.

Market returns:
{features['returns']}

Detected signals:
{signals}

Alerts:
{alerts}

Key news:
{news}

Output STRICTLY in the following format only. Do not include Detailed Analysis, Conclusion, Recommendations, or Disclaimer sections:

## 🚨 Alerts
- ...

## 📊 Trading Signals
- ...

## Executive Summary
- ...

## Key Drivers
- ...

## Risks
- ...

## Outlook
- ...
"""

In [29]:
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
input_length = inputs["input_ids"].shape[1]
output = model.generate(**inputs, max_new_tokens=1000)

# Decode only the generated tokens, skipping the prompt
summary = tokenizer.decode(output[0][input_length:], skip_special_tokens=True)

open("report.txt", "w").write(summary)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


1470

In [19]:
import os, json
from IPython.display import display, Markdown
from sendgrid import SendGridAPIClient
from sendgrid.helpers.mail import Mail

In [36]:
os.environ["SENDGRID_API"] = "SG.TT-***" 

In [37]:
sendgrid_key = os.environ.get('SENDGRID_API')
sendgrid_key

'SG.TT-***'

In [30]:
plain_text_content=open("report.txt").read()
display(Markdown(plain_text_content))


## Market Summary
- SPY: 0.0026305975119209624
- QQQ: 0.0048802302740798
- DIA: 0.001275794427151311
- NVDA: -0.00704076266961684
- SNOW: 0.001146859025407787

## Trading Signals
- broad_bullish: medium
- negative_news_pressure: medium

## Key Drivers
- Japan's Mitsui OSK shares soar to record highs as activist investor Elliott builds 'significant' stake
- South Korea's stocks lead gains in Asia as investors assess Japan trade data, await Fed rate verdict
- Elon Musk and SEC in talks to settle lawsuit over Twitter deal
- The Fed issues its latest interest rate decision Wednesday. Here's what to expect
- Lululemon reports weak guidance as proxy battle, tariffs weigh on bottom line
- Orlando Bravo pushes back on private markets criticism: 'Everybody's extremely comfortable'
- Global hedge funds suffer worst losses since 'liberation day' on Iran war turmoil
- Nvidia CEO Jensen Huang says OpenClaw is 'definitely the next ChatGPT'
- Jensen Huang says Nvidia has received orders from China and is 'restarting our manufacturing'
- Nvidia’s big GTC showcase barely budged the stock. Is that a problem?

## Risks
- Middle East war continues to escalate, keeping investors on edge
- Iran war continues, causing market volatility

## Outlook
- Market is showing a medium level of bullishness
- However, there is a medium level of negative news pressure due to the ongoing Middle East war and Iran war
- Investors should be cautious and closely monitor the situation.

In [31]:
alerts = json.load(open("data/alerts.json"))

if len(alerts) > 0:
    subject = "🚨 MARKET ALERT - ACTION REQUIRED"
else:
    subject = "📊 Daily Market Summary"

In [32]:
content = open("report.txt").read()

html_content = f"""
<h2>📊 Market Summary</h2>
<pre>{content}</pre>
"""

In [34]:
message = Mail(
    from_email='david.guo@utsa.edu',
    to_emails='david.guo@utsa.edu',
    subject=subject,
    html_content=html_content
)

# Recommended: Use environment variables
sg = SendGridAPIClient(api_key=sendgrid_key)

sg.send(message)

# set up running schedule

In [ ]:
#bash
>crontab -e

In [ ]:
>0 6 * * * /usr/bin/python3 /path/to/send_email.py >> /path/to/logs/email.log 2>&1